In [22]:
# Imports section
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import PolynomialFeatures

## Part 1. Loading the dataset

In [23]:
# Using pandas load the dataset (load remotely, not locally)
df = pd.read_csv('science_data_large.csv')

# Output the first 15 rows of the data
print(df.head(15), '\n')

# Display a summary of the table information (number of datapoints, etc.)
print(df.describe(include="all"), '\n')

# Checking for null values in the dataset
print(df.isnull().sum())

    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04 

       Temperature °C     Mols KCL     Size nm^3
count     1000.000000  1000.000000  1.000000e+03
mean       500.500000   471.530000  5.086111e+05
std        288.819436   288.482872  4.474838e+05
min          1.000000     1.000000  1.611429e+01
25%        250.750000   226.750000  1.298267e+05
50%        500.5

## Part 2. Splitting the dataset

In [24]:
# Take the pandas dataset and split it into our features (X) and label (y)
X = df.drop('Size nm^3', axis = 1) # Features (X)
y = df['Size nm^3'] # label (y)

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, random_state = 0)

## Part 3. Perform a Linear Regression

In [25]:
# Use sklearn to train a model on the training set
regressor = LinearRegression()
regressor.fit(X_train, y_train)

# Create a sample datapoint and predict the output of that sample with the trained model
sample_datapoint = X.iloc[4].values.reshape(1, -1)
predicted_size = regressor.predict(sample_datapoint)
print(f"The predicted size for the sample datapoint is: {predicted_size[0]:.2f} nm^3")

# Obtaining the score for the model on unseen data (10% testing data)
test_score = regressor.score(X_test, y_test)
print(f"The R² score on the test set is: {test_score:.5f}")

# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
coefficients = regressor.coef_
intercept = regressor.intercept_

# Displaying the coefficients and intercept
print(f"Intercept (β0): {intercept:.5f}")
print(f"Coefficient for Temperature (β1): {coefficients[0]:.5f}")
print(f"Coefficient for Mols KCL (β2): {coefficients[1]:.5f}")

The predicted size for the sample datapoint is: 395890.94 nm^3
The R² score on the test set is: 0.87616
Intercept (β0): -400305.91333
Coefficient for Temperature (β1): 863.58109
Coefficient for Mols KCL (β2): 1006.12742


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


# Equation:
$h(x) = {-400305.91333} + \beta_1 \cdot \text{863.58109} + \beta_2 \cdot \text{1006.12742}$

The **R² score** (coefficient of determination) is a measure that indicates how well the regression model predicts the target variable, in this case, `Size nm³`. The R² score ranges from 0 to 1, where 1 represents a perfect fit, meaning the model explains 100% of the variability in the target variable, and 0 means the model does not explain any of the variability, performing no better than a simple mean prediction. Negative values indicate that the model performs worse than a horizontal line representing the mean of the target values.

In this instance, the R² score on the test set is **0.87616**, which means the model explains approximately **87.6% of the variance** in `Size nm³` based on `Temperature` and `Mols KCL`. This suggests that the model captures a strong relationship between the predictors and the target variable, making it a good fit. However, the remaining **12.4%** of unexplained variability indicates there is still room for improvement in the model's predictive performance.

## Part 4. Use Cross Validation

In [26]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
kf = KFold(n_splits=10, shuffle=True, random_state=42) # Using KFold with shuffling to ensure that the data is shuffled before splitting into folds
cv_scores = cross_val_score(regressor, X_train, y_train, cv=kf, scoring='r2') # Using cross_val_score to perform 5-fold cross-validation
print(f"Cross-validation R² scores: {cv_scores}")
print(f"Mean R² score: {cv_scores.mean():.5f}")

# After cross-validation, fitting the model to the full training set
regressor.fit(X_train, y_train)

# Creating a sample datapoint and predict the output of that sample with the trained model
sample_datapoint_cv = X.iloc[4].values.reshape(1, -1)
predicted_size_cv = regressor.predict(sample_datapoint_cv)
print(f"The predicted size for the sample datapoint is: {predicted_size_cv[0]:.2f} nm^3")

# Evaluating the model on the unseen test set (10% partitioned data)
test_score = regressor.score(X_test, y_test)
print(f"The R² score on the test set is: {test_score:.5f}")

Cross-validation R² scores: [0.86533191 0.84675582 0.84495966 0.85989746 0.84991245 0.86533485
 0.85669548 0.86379613 0.83182178 0.85643631]
Mean R² score: 0.85409
The predicted size for the sample datapoint is: 395890.94 nm^3
The R² score on the test set is: 0.87616


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


# Report on their finding and their significance
The cross-validation results provide a robust assessment of the model's performance across multiple random splits of the training data. The **R² scores** from 10-fold cross-validation range between **0.8318** and **0.8653**, indicating that the model consistently explains a high proportion of the variance in the data across different subsets. The **mean R² score** from cross-validation is **0.8541**, which suggests the model generalizes well, with an average of **85.4%** of the variance explained. After training the model on the full dataset, the model predicts the size for a sample datapoint to be **395,890.94 nm³**. Additionally, the **R² score** on the unseen test set is **0.8762**, meaning that the model explains approximately **87.6%** of the variance in the test data. These results indicate that the model performs well both during cross-validation and when evaluated on new, unseen data, making it a reliable predictor of `Size nm³` based on `Temperature` and `Mols KCL`.

## Part 5. Using Polynomial Regression

In [27]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2

# Apply PolynomialFeatures of degree 2
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# Fit a linear regression model to the polynomial-transformed training data
poly_regressor = LinearRegression()
poly_regressor.fit(X_train_poly, y_train)

# Make a prediction for a sample datapoint
sample_datapoint = X.iloc[4].values.reshape(1, -1)
sample_datapoint_poly = poly.transform(sample_datapoint)
predicted_size_poly = poly_regressor.predict(sample_datapoint_poly)
print(f"The predicted size for the sample datapoint using polynomial regression is: {predicted_size_poly[0]:.2f} nm^3")

# Evaluate the model's performance on training and test sets
train_score_poly = poly_regressor.score(X_train_poly, y_train)
test_score_poly = poly_regressor.score(X_test_poly, y_test)
print(f"R² score for the polynomial regression model on training set: {train_score_poly:.5f}")
print(f"R² score for the polynomial regression model on test set: {test_score_poly:.5f}")

# Extract the coefficients and intercept
coefficients = poly_regressor.coef_
intercept = poly_regressor.intercept_

# Get the names of the polynomial features (including interaction and squared terms)
poly_feature_names = poly.get_feature_names_out(['Temperature °C', 'Mols KCL'])

# Display the coefficients and corresponding polynomial terms
print("The polynomial regression equation is:")
terms = []
for i, coef in enumerate(coefficients):
    terms.append(f"{coef:.5f} * {poly_feature_names[i]}")

# Write the full equation for h(x)
equation = "h(x) = " + f"{intercept:.5f} + " + " + ".join(terms)
print(equation)

The predicted size for the sample datapoint using polynomial regression is: 43257.26 nm^3
R² score for the polynomial regression model on training set: 1.00000
R² score for the polynomial regression model on test set: 1.00000
The polynomial regression equation is:
h(x) = 0.00001 + 0.00000 * 1 + 12.00000 * Temperature °C + -0.00000 * Mols KCL + -0.00000 * Temperature °C^2 + 2.00000 * Temperature °C Mols KCL + 0.02857 * Mols KCL^2


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but PolynomialFeatures was fitted with feature names
  warnings.warn(


# Equation
$$
h(x) = 0.00001 + (0.00000) \cdot 1 + (12.00000) \cdot \text{Temperature} \, ^\circ C + (-0.00000) \cdot \text{Mols KCL} + \\ (-0.00000) \cdot \text{Temperature} \, ^\circ C^2 + (2.00000) \cdot \text{Temperature} \, ^\circ C \cdot \text{Mols KCL} + 0.02857 \cdot \text{Mols KCL}^2
$$

$$
h(x) = 0.00001 + (12.00000) \cdot \text{Temperature} \, ^\circ C + (2.00000) \cdot \text{Temperature} \, ^\circ C \cdot \text{Mols KCL} + 0.02857 \cdot \text{Mols KCL}^2
$$



The polynomial regression model was applied using a second-degree polynomial transformation on the dataset. The model predicted a sample data point with a value of 43,257.26 nm³, which closely matches the expected size. The R² score for both the training and test sets is 1.00000, indicating that the model perfectly fits the data and captures all of the variance in the dataset. This suggests that the model is highly accurate for the given data. The h(x) equation shows that Temperature and Mols KCL have significant interaction effects on the predicted size, especially the terms involving the interaction between Temperature and Mols KCL, as well as the squared term for Mols KCL. The perfect score of 1 on both training and test sets suggests a near-perfect model fit.